In [ ]:
class Fenwick(object):
    def __init__(self, n):
        self.n = n
        self.bit = [0] * (n + 2)

    def add(self, index, value):
        while index <= self.n:
            self.bit[index] += value
            index += index & -index

    def sum(self, index):
        total = 0
        while index > 0:
            total += self.bit[index]
            index -= index & -index
        return total

    def kth(self, k):
        index = 0
        bit_mask = 1

        while bit_mask * 2 <= self.n:
            bit_mask *= 2

        while bit_mask:
            next_index = index + bit_mask
            if next_index <= self.n and self.bit[next_index] < k:
                index = next_index
                k -= self.bit[next_index]
            bit_mask //= 2

        return index + 1


class SegmentTree(object):
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (2 * n)

    def update(self, index, value):
        index += self.n - 1
        self.tree[index] = value

        index //= 2
        while index:
            self.tree[index] = max(self.tree[2 * index], self.tree[2 * index + 1])
            index //= 2

    def query(self, left, right):
        if left > right:
            return 0

        left += self.n - 1
        right += self.n - 1
        answer = 0

        while left <= right:
            if left % 2 == 1:
                answer = max(answer, self.tree[left])
                left += 1

            if right % 2 == 0:
                answer = max(answer, self.tree[right])
                right -= 1

            left //= 2
            right //= 2

        return answer


class Solution(object):
    def getResults(self, queries):
        max_x = 0

        for query in queries:
            max_x = max(max_x, query[1])

        size = max_x + 2

        fenwick = Fenwick(size)
        segtree = SegmentTree(size)

        results = []

        for query in queries:
            if query[0] == 1:
                x = query[1]

                count_before_or_at_x = fenwick.sum(x)
                total_obstacles = fenwick.sum(size)

                prev_obstacle = 0
                if count_before_or_at_x > 0:
                    prev_obstacle = fenwick.kth(count_before_or_at_x)

                next_obstacle = None
                if count_before_or_at_x < total_obstacles:
                    next_obstacle = fenwick.kth(count_before_or_at_x + 1)

                fenwick.add(x, 1)

                segtree.update(x, x - prev_obstacle)

                if next_obstacle is not None:
                    segtree.update(next_obstacle, next_obstacle - x)

            else:
                x = query[1]
                sz = query[2]

                count_before_or_at_x = fenwick.sum(x)

                prev_obstacle = 0
                if count_before_or_at_x > 0:
                    prev_obstacle = fenwick.kth(count_before_or_at_x)

                partial_gap = x - prev_obstacle
                max_full_gap = segtree.query(1, x)

                best_gap = max(partial_gap, max_full_gap)

                results.append(best_gap >= sz)

        return results